# OKAFF TCPD Oracle results

In [1]:
from pathlib import Path
import hashlib
import json
import math
import sys

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

START_DIR = Path.cwd().resolve()
DATA_MARKERS = ('metrics.py', 'annotations.json', 'datasets')


def is_data_dir(path):
    return all((path / marker).exists() for marker in DATA_MARKERS)


data_candidates = [START_DIR, *START_DIR.parents]
try:
    data_candidates.extend(
        path for path in START_DIR.iterdir() if path.is_dir()
    )
except OSError:
    pass

HERE = next((path for path in data_candidates if is_data_dir(path)), None)
if HERE is None:
    raise FileNotFoundError(
        'Could not locate the real-data folder containing '
        + ', '.join(DATA_MARKERS)
        + f'. Current working directory: {START_DIR}'
    )

CODE_ROOT = next(
    (
        parent
        for parent in (HERE, *HERE.parents)
        if (parent / 'src' / 'kerneldetector.py').is_file()
        and (parent / 'src' / 'adaptivethreholds.py').is_file()
    ),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError('Could not locate the project src directory')

SRC_DIR = CODE_ROOT / 'src'
for module_dir in (HERE, SRC_DIR):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import kerneldetector
import adaptivethreholds

if Path(kerneldetector.__file__).resolve().parent != SRC_DIR.resolve():
    raise RuntimeError('Restart the kernel so kerneldetector loads from src')
if Path(adaptivethreholds.__file__).resolve().parent != SRC_DIR.resolve():
    raise RuntimeError('Restart the kernel so adaptivethreholds loads from src')

from kerneldetector import OKAFF, estimate_gaussian_gamma
from adaptivethreholds import OKAFFAdaptiveThreshold
from metrics import f_measure

REFERENCE_SIZE = 50
FEATURES = 500
LAMBDA_MAX = 0.999
LAMBDA_MIN = 0.001
REFERENCE_THRESHOLD_ALPHA = 0.10
SEED = 12345
THEORY_BANDWIDTH_SEED = 12345
MATCH_MARGIN = 5
INITIALIZATION = 'theory_A_SV'  # change to 'zero' to select zero initialization
QUALITY_CONTROL = {f'quality_control_{i}' for i in range(1, 6)}
TABLE2_EXCLUDED = QUALITY_CONTROL | {'uk_coal_employ'}
DATASET_DIR = HERE / 'datasets'
ANNOTATIONS_FILE = HERE / 'annotations.json'
RESULTS = (
    HERE
    / 'results'
    / 'OKAFF_kerneldetector_TCPD'
)
RESULTS.mkdir(parents=True, exist_ok=True)


def load_tcpd_dataset(path):
    with path.open() as stream:
        data = json.load(stream)
    if data['time']['index'] != list(range(data['n_obs'])):
        raise ValueError(f'{path}: non-consecutive time index')
    matrix = np.empty((data['n_obs'], data['n_dim']), dtype=float)
    for column, series in enumerate(data['series']):
        matrix[:, column] = [
            np.nan if value is None else value for value in series['raw']
        ]
    return data, matrix


def estimate_sigmasq(reference):
    gamma = estimate_gaussian_gamma(np.asarray(reference), max_len=len(reference))
    if not np.isfinite(gamma) or gamma <= 0:
        raise ValueError(f'Non-positive Gaussian gamma={gamma}')
    return 1.0 / (2.0 * gamma)


def standard_normal_bandwidth(dimension, reference_size, seed):
    rng = np.random.default_rng(seed + dimension)
    reference = rng.standard_normal((reference_size, dimension))
    return estimate_sigmasq(reference)


def make_rff(dimension, features, sigmasq, seed):
    rng = np.random.default_rng(seed)
    frequencies = rng.normal(
        scale=1 / math.sqrt(sigmasq), size=(features, dimension)
    )
    scale = 1 / math.sqrt(features)

    def feature_map(sample):
        projection = frequencies @ np.asarray(sample, dtype=float)
        return scale * np.concatenate((np.cos(projection), np.sin(projection)))

    return feature_map


def stable_seed(base_seed, dataset):
    digest = hashlib.sha256(dataset.encode()).digest()
    return (base_seed + int.from_bytes(digest[:8], 'little')) % (2**63 - 1)


def logdet_spd(matrix):
    sign, value = np.linalg.slogdet(matrix)
    if sign <= 0:
        raise ValueError('Expected a positive-definite matrix')
    return float(value)


def gaussian_kernel_terms(dimension, sigmasq):
    identity = np.eye(dimension)
    ld1 = logdet_spd(identity + identity / sigmasq)
    ld2 = logdet_spd(identity + 2 * identity / sigmasq)
    ld3 = logdet_spd(identity + 3 * identity / sigmasq)
    ld4 = logdet_spd(identity + 4 * identity / sigmasq)
    theta = math.exp(-0.5 * ld2)
    term13 = math.exp(-0.5 * (ld1 + ld3))
    term2 = math.exp(-ld2)
    term4 = math.exp(-0.5 * ld4)
    return (
        theta,
        max(0.0, term13 - term2),
        max(0.0, term4 - 2 * term13 + term2),
    )


def theoretical_center_sv(band_lambda, dimension, sigmasq):
    theta, eta_k, zeta_k = gaussian_kernel_terms(dimension, sigmasq)
    lam = band_lambda
    delta = (1 - lam) / (1 + lam)
    rho = (1 - lam)**2 / (1 + lam + lam**2)
    kappa = (1 - lam)**3 / (1 + lam + lam**2 + lam**3)
    center = delta + (1 - delta) * theta
    variance = (
        4 * (delta - 2 * rho + kappa) * eta_k
        + 2 * (delta**2 - kappa) * zeta_k
    )
    return center, math.sqrt(max(0.0, variance))



def make_two_rate_threshold(
    quantile,
    theory_A,
    theory_SV,
):
    """Create the threshold for zero-based data indices 0,...,49 as reference."""
    return OKAFFAdaptiveThreshold(
        alpha=REFERENCE_THRESHOLD_ALPHA,
        quantile=float(quantile),
        t0=REFERENCE_SIZE,
        initialization=INITIALIZATION,
        theory_A=theory_A if INITIALIZATION == 'theory_A_SV' else None,
        theory_SV=theory_SV if INITIALIZATION == 'theory_A_SV' else None,
    )


def update_two_rate_threshold(
    threshold,
    index,
    statistic,
    monitoring_alpha,
):
    if index == REFERENCE_SIZE:
        threshold.alpha = float(monitoring_alpha)
    return threshold.update(float(statistic))



print('Experiment folder:', HERE)
print('Results folder:', RESULTS)


## Oracle grid search and tables


In [2]:
ORACLE_ETA_UNIVARIATE_GRID = (1e-5, 1e-4, 1e-3, 1e-2)
ORACLE_ETA_MULTIVARIATE_GRID = (1e-5, 1e-4, 1e-3, 1e-2)
ORACLE_THRESHOLD_ALPHA_GRID = (0.005, 0.01, 0.05, 0.10, 0.20)
ORACLE_THRESHOLD_QUANTILE_GRID = (0.95, 0.975, 0.995, 0.9975)


def calculate_okaff_statistics(matrix, name, eta):
    detector_sigmasq = estimate_sigmasq(matrix[:REFERENCE_SIZE])
    feature_map = make_rff(
        matrix.shape[1],
        FEATURES,
        detector_sigmasq,
        stable_seed(SEED, name),
    )
    detector = OKAFF(
        lambda0=LAMBDA_MAX,
        lambda1=LAMBDA_MAX,
        eta=eta,
        feat_func=feature_map,
        clip=(LAMBDA_MIN, LAMBDA_MAX),
        thresholding_method='fixed',
        fixed_threshold=np.inf,
        store_values=False,
    )
    statistics = np.empty(len(matrix))
    for index, sample in enumerate(matrix):
        detector.update(sample)
        statistics[index] = float(detector.statistic)

    theory_sigmasq = standard_normal_bandwidth(
        matrix.shape[1], REFERENCE_SIZE, THEORY_BANDWIDTH_SEED
    )
    theory_A, theory_SV = theoretical_center_sv(
        LAMBDA_MAX, matrix.shape[1], theory_sigmasq
    )
    return statistics, theory_A, theory_SV


def oracle_alarms(statistics, alpha, quantile, theory_A, theory_SV):
    threshold = make_two_rate_threshold(
        quantile=quantile,
        theory_A=theory_A,
        theory_SV=theory_SV,
    )
    alarms = []
    for index, statistic in enumerate(statistics):
        alarm = update_two_rate_threshold(
            threshold,
            index,
            statistic,
            monitoring_alpha=alpha,
        )
        if index >= REFERENCE_SIZE and alarm:
            alarms.append(index)
    return alarms


with ANNOTATIONS_FILE.open() as stream:
    annotations = json.load(stream)

selected = sorted(set(annotations) - TABLE2_EXCLUDED)
matrices = {}
dataset_types = {}
for name in selected:
    data, matrix = load_tcpd_dataset(DATASET_DIR / f'{name}.json')
    if len(matrix) > REFERENCE_SIZE:
        matrices[name] = matrix
        dataset_types[name] = (
            'multivariate' if data['n_dim'] > 1 else 'univariate'
        )

oracle_rows = []
for dataset_number, name in enumerate(sorted(matrices), 1):
    matrix = matrices[name]
    dimensionality = dataset_types[name]
    eta_grid = (
        ORACLE_ETA_MULTIVARIATE_GRID
        if dimensionality == 'multivariate'
        else ORACLE_ETA_UNIVARIATE_GRID
    )
    evaluation_annotations = {
        annotator: [
            point
            for point in points
            if point >= REFERENCE_SIZE - MATCH_MARGIN
        ]
        for annotator, points in annotations[name].items()
    }
    print(f'[Oracle {dataset_number:02d}/{len(matrices):02d}] {name}', flush=True)

    for eta in eta_grid:
        statistics, theory_A, theory_SV = calculate_okaff_statistics(
            matrix, name, eta
        )
        for alpha in ORACLE_THRESHOLD_ALPHA_GRID:
            for quantile in ORACLE_THRESHOLD_QUANTILE_GRID:
                predictions = oracle_alarms(
                    statistics, alpha, quantile, theory_A, theory_SV
                )
                f1 = f_measure(evaluation_annotations, predictions)
                oracle_rows.append({
                    'dataset': name,
                    'dimensionality': dimensionality,
                    'eta': eta,
                    'reference_threshold_alpha': REFERENCE_THRESHOLD_ALPHA,
                    'threshold_alpha': alpha,
                    'threshold_quantile': quantile,
                    'f1': f1,
                })

oracle_grid = pd.DataFrame(oracle_rows)
oracle_f1 = oracle_grid.loc[
    oracle_grid.groupby('dataset')['f1'].idxmax()
].sort_values('dataset').reset_index(drop=True)

oracle_summary = (
    oracle_f1.groupby('dimensionality', as_index=False)
    .agg(**{
        'Oracle F1': ('f1', 'mean'),
        'N datasets': ('dataset', 'nunique'),
    })
    .rename(columns={'dimensionality': 'data type'})
    [['data type', 'Oracle F1', 'N datasets']]
)

oracle_f1_table = oracle_f1[[
    'dataset',
    'dimensionality',
    'eta',
    'reference_threshold_alpha',
    'threshold_alpha',
    'threshold_quantile',
    'f1',
]].rename(columns={
    'dimensionality': 'data type',
    'reference_threshold_alpha': 'reference alpha',
    'threshold_alpha': 'monitoring alpha',
    'threshold_quantile': 'q',
    'f1': 'Oracle F1',
})

oracle_summary.to_csv(RESULTS / 'okaff_oracle_summary.csv', index=False)
oracle_f1_table.to_csv(
    RESULTS / 'okaff_oracle_f1_by_dataset.csv', index=False
)

display(Markdown('### Overall Oracle results'))
display(oracle_summary.round({'Oracle F1': 3}))

aggregate_excluded_datasets = {'gdp_iran', 'gdp_japan', 'ozone', 'robocalls'}
oracle_summary_excluding = (
    oracle_f1.loc[~oracle_f1['dataset'].isin(aggregate_excluded_datasets)]
    .groupby('dimensionality', as_index=False)
    .agg(**{
        'Oracle F1': ('f1', 'mean'),
        'N datasets': ('dataset', 'nunique'),
    })
    .rename(columns={'dimensionality': 'data type'})
    [['data type', 'Oracle F1', 'N datasets']]
)
display(Markdown(
    '### Oracle results excluding `gdp_iran`, `gdp_japan`, `ozone`, and `robocalls`'
))
display(oracle_summary_excluding.round({'Oracle F1': 3}))

display(Markdown('### Oracle-F1: maximum F1 per dataset'))
with pd.option_context('display.max_rows', None):
    display(oracle_f1_table.round(3))


[Oracle 01/32] apple
[Oracle 02/32] bank
[Oracle 03/32] bee_waggle_6
[Oracle 04/32] bitcoin
[Oracle 05/32] brent_spot
[Oracle 06/32] businv
[Oracle 07/32] children_per_woman
[Oracle 08/32] co2_canada
[Oracle 09/32] construction
[Oracle 10/32] gdp_argentina
[Oracle 11/32] gdp_iran
[Oracle 12/32] gdp_japan
[Oracle 13/32] global_co2
[Oracle 14/32] homeruns
[Oracle 15/32] iceland_tourism
[Oracle 16/32] jfk_passengers
[Oracle 17/32] lga_passengers
[Oracle 18/32] measles
[Oracle 19/32] nile
[Oracle 20/32] occupancy
[Oracle 21/32] ozone
[Oracle 22/32] ratner_stock
[Oracle 23/32] robocalls
[Oracle 24/32] run_log
[Oracle 25/32] scanline_126007
[Oracle 26/32] scanline_42049
[Oracle 27/32] seatbelts
[Oracle 28/32] shanghai_license
[Oracle 29/32] unemployment_nl
[Oracle 30/32] us_population
[Oracle 31/32] usd_isk
[Oracle 32/32] well_log


### Overall Oracle results

,data type,Oracle F1,N datasets
0,multivariate,0.767,4
1,univariate,0.815,28


### Oracle results excluding `gdp_iran`, `gdp_japan`, `ozone`, and `robocalls`

,data type,Oracle F1,N datasets
0,multivariate,0.767,4
1,univariate,0.784,24


### Oracle-F1: maximum F1 per dataset

,dataset,data type,eta,reference alpha,monitoring alpha,q,Oracle F1
0,apple,multivariate,0.010,0.1,0.100,0.975,0.634
1,bank,univariate,0.000,0.1,0.005,0.950,1.000
2,bee_waggle_6,multivariate,0.000,0.1,0.005,0.995,0.929
3,bitcoin,univariate,0.001,0.1,0.200,0.975,0.479
4,brent_spot,univariate,0.001,0.1,0.200,0.950,0.544
5,businv,univariate,0.010,0.1,0.050,0.995,0.919
6,children_per_woman,univariate,0.000,0.1,0.005,0.995,0.507
7,co2_canada,univariate,0.001,0.1,0.200,0.950,0.503
8,construction,univariate,0.000,0.1,0.005,0.998,0.696
9,gdp_argentina,univariate,0.000,0.1,0.005,0.995,0.889
